# Notebook 1 – Sentinel-1 Data Export (Google Earth Engine)

**Goal:** create the training dataset for land use / land cover (LULC) segmentation using **only Sentinel-1 radar data** as input.

**Output:** 40 GeoTIFF files (10 regions × 4 months) in Google Drive → `LULC_S1`.
Each file has 4 bands: `VV`, `VH`, `VV_VH` (input, in dB) and `label` (ESA WorldCover class 0–6).

## Step 1 – Connect to Google Earth Engine

**What:** log in and connect to the Earth Engine project `lulc-bharat-2026`.

**Why:** Earth Engine stores the complete Sentinel-1 and ESA WorldCover archives and processes them on Google's servers, so no large raw files have to be downloaded.

In [1]:
import ee, time

ee.Authenticate()
ee.Initialize(project='lulc-bharat-2026')
print('Earth Engine connected')

Earth Engine connected


## Step 2 – Regions and data split

**What:** define 10 regions in Europe (16 km × 16 km each) and assign each region to train (7), validation (1) or test (2).

**Why:**
- The regions cover different landscapes (large cities, cropland, forest, dry land, water, wetland), so the model learns many land-cover types.
- The split is done **by region**, not by random patches. Neighbouring patches look almost the same, so a random split would give a too optimistic test score. The test regions Lyon and Hamburg are completely unseen during training.
- Four months of 2021 (January, April, July, October) are used, one per season, so the model learns how the radar signal changes over the year.
- Mannheim is not included because it is used later for the case study.

In [3]:
# name: (longitude, latitude, split)
REGIONS = {
    'paris':        ( 2.35, 48.86, 'train'),
    'lelystad':     ( 5.47, 52.52, 'train'),
    'blackforest':  ( 8.20, 48.00, 'train'),
    'povalley':     (10.00, 45.10, 'train'),   # intensive cropland
    'madrid':       (-3.70, 40.42, 'train'),   # dry city, bare land
    'danubedelta':  (29.00, 45.15, 'train'),   # wetland, water
    'warsaw':       (21.01, 52.23, 'train'),   # city, forest, cropland
    'munich':       (11.58, 48.14, 'val'),     # validation (model selection)
    'lyon':         ( 4.83, 45.76, 'test'),    # test - never seen in training
    'hamburg':      ( 9.99, 53.55, 'test'),    # test - never seen in training
}

HALF_SIZE_M = 8000                         # 8 km -> 16 km square
MONTHS = ['2021-01-01', '2021-04-01', '2021-07-01', '2021-10-01']   # winter, spring, summer, autumn

def region_geom(lon, lat):
    return ee.Geometry.Point([lon, lat]).buffer(HALF_SIZE_M).bounds()

## Step 3 – Sentinel-1 preprocessing

**What:** build one monthly composite image (VV, VH, VV−VH) for a region and month.

**Already done by ESA SNAP in the Earth Engine collection `COPERNICUS/S1_GRD`:** thermal noise removal, radiometric calibration (σ⁰), terrain correction and conversion to dB.

**Added in this project:**
- **Border-noise mask:** values below −30 dB at image edges are artefacts and are removed.
- **Speckle filter (3 × 3 median):** SAR images are grainy; a median filter reduces this noise and keeps edges sharp.
- **Monthly median composite:** all scenes of one month are combined pixel by pixel. This reduces speckle further and gives exactly one image per month, the same format the prototype uses.
- **VV − VH band:** in dB this equals the ratio VV/VH and helps to separate vegetation from built-up and bare areas.

In [4]:
def s1_monthly_composite(region, start_date):
    """Return a 3-band image (VV, VH, VV_VH) = median of all Sentinel-1 IW scenes in one month."""
    start = ee.Date(start_date)
    end = start.advance(1, 'month')

    def clean(img):
        img = img.updateMask(img.select('VV').gt(-30))           # removing border noise
        return img.focal_median(radius=1, kernelType='square', units='pixels')   # 3x3 speckle filter

    col = (ee.ImageCollection('COPERNICUS/S1_GRD')
           .filterBounds(region)
           .filterDate(start, end)
           .filter(ee.Filter.eq('instrumentMode', 'IW'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
           .select(['VV', 'VH'])
           .map(clean))

    img = col.median()
    ratio = img.select('VV').subtract(img.select('VH')).rename('VV_VH')
    return img.addBands(ratio), col.size()

## Step 4 – Labels (ESA WorldCover 2021)

**What:** load ESA WorldCover 2021 (10 m) and merge its 11 classes into 7 classes.

| ID | Class | WorldCover codes |
|---|---|---|
| 0 | Tree | 10 |
| 1 | Grass/Shrub | 20, 30 |
| 2 | Cropland | 40 |
| 3 | Built-up | 50 |
| 4 | Bare/Sparse | 60, 70, 100 |
| 5 | Water | 80 |
| 6 | Wetland | 90, 95 |

**Why:** WorldCover has the same 10 m resolution as Sentinel-1 and describes the same year (2021). Some classes look almost identical in radar (shrub vs. grass) or do not exist in Europe (mangroves), so they are merged. Pixels without a label get the value 255 and are ignored later.

In [5]:
CLASS_NAMES = ['Tree', 'Grass/Shrub', 'Cropland', 'Built-up', 'Bare/Sparse', 'Water', 'Wetland']

worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
label = (worldcover
         .remap([10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
                [ 0,  1,  1,  2,  3,  4,  4,  5,  6,  6,   4])
         .rename('label'))

## Step 5 – Check the number of Sentinel-1 scenes

**What:** count the Sentinel-1 scenes for every region and month.

**Why:** each monthly composite should be based on several scenes. The output shows 14–32 scenes per region and month, so all composites are reliable.

In [6]:
for name, (lon, lat, split) in REGIONS.items():
    geom = region_geom(lon, lat)
    counts = [s1_monthly_composite(geom, m)[1].getInfo() for m in MONTHS]
    print(f'{name:12s} {split:5s} scenes per month: {counts}')

paris        train scenes per month: [15, 15, 16, 14]
lelystad     train scenes per month: [25, 25, 25, 26]
blackforest  train scenes per month: [22, 22, 22, 22]
povalley     train scenes per month: [19, 15, 19, 17]
madrid       train scenes per month: [17, 15, 15, 15]
danubedelta  train scenes per month: [19, 21, 19, 22]
warsaw       train scenes per month: [19, 20, 21, 20]
munich       val   scenes per month: [20, 27, 31, 32]
lyon         test  scenes per month: [19, 20, 19, 20]
hamburg      test  scenes per month: [22, 23, 21, 24]


## Step 6 – Export to Google Drive

**What:** stack the composite and the label into one image and export it as a GeoTIFF.

**Settings:** projection EPSG:3035 (equal-area grid for Europe), 10 m pixels. Labels use nearest-neighbour resampling, so class IDs are never mixed. Missing SAR values are set to −50 and missing labels to 255.

**Output:** 40 files in Google Drive → `LULC_S1`, named `<split>_<region>_<month>.tif` (for example `train_paris_2021-07.tif`).

In [7]:
tasks = []
for name, (lon, lat, split) in REGIONS.items():
    geom = region_geom(lon, lat)
    for m in MONTHS:
        s1, _ = s1_monthly_composite(geom, m)
        stack = (s1.unmask(-50)
                   .addBands(label.unmask(255))
                   .clip(geom)
                   .toFloat())
        file_name = f'{split}_{name}_{m[:7]}'
        task = ee.batch.Export.image.toDrive(
            image=stack,
            description=file_name,
            folder='LULC_S1',
            fileNamePrefix=file_name,
            region=geom,
            scale=10,
            crs='EPSG:3035',
            maxPixels=1e9)
        task.start()
        tasks.append(task)
print(f'Started {len(tasks)} export tasks')

Started 40 export tasks


## Step 7 – Wait for the exports

**What:** check the status of all 40 export tasks until they are completed.

**Result:** all exports completed; the dataset is ready for Notebook 2.

In [8]:
while True:
    states = [t.status()['state'] for t in tasks]
    summary = {s: states.count(s) for s in set(states)}
    print(summary)
    if all(s in ('COMPLETED', 'FAILED', 'CANCELLED') for s in states):
        break
    time.sleep(60)

for t in tasks:
    st = t.status()
    if st['state'] != 'COMPLETED':
        print(st['description'], st['state'], st.get('error_message', ''))

{'COMPLETED': 3, 'READY': 35, 'RUNNING': 2}
{'COMPLETED': 3, 'READY': 35, 'RUNNING': 2}
{'COMPLETED': 4, 'READY': 34, 'RUNNING': 2}
{'COMPLETED': 5, 'READY': 33, 'RUNNING': 2}
{'COMPLETED': 5, 'READY': 33, 'RUNNING': 2}
{'COMPLETED': 5, 'READY': 33, 'RUNNING': 2}
{'COMPLETED': 6, 'READY': 32, 'RUNNING': 2}
{'COMPLETED': 6, 'READY': 32, 'RUNNING': 2}
{'COMPLETED': 7, 'READY': 31, 'RUNNING': 2}
{'COMPLETED': 8, 'READY': 30, 'RUNNING': 2}
{'COMPLETED': 9, 'READY': 29, 'RUNNING': 2}
{'COMPLETED': 9, 'READY': 29, 'RUNNING': 2}
{'COMPLETED': 9, 'READY': 29, 'RUNNING': 2}
{'COMPLETED': 9, 'READY': 29, 'RUNNING': 2}
{'COMPLETED': 10, 'READY': 28, 'RUNNING': 2}
{'COMPLETED': 11, 'READY': 27, 'RUNNING': 2}
{'COMPLETED': 11, 'READY': 27, 'RUNNING': 2}
{'COMPLETED': 12, 'READY': 26, 'RUNNING': 2}
{'COMPLETED': 14, 'READY': 24, 'RUNNING': 2}
{'COMPLETED': 14, 'READY': 24, 'RUNNING': 2}
{'COMPLETED': 14, 'READY': 24, 'RUNNING': 2}
{'COMPLETED': 15, 'READY': 24, 'RUNNING': 1}
{'COMPLETED': 15, 'READY

{'COMPLETED': 23, 'READY': 15, 'RUNNING': 2}
{'COMPLETED': 23, 'READY': 15, 'RUNNING': 2}
{'COMPLETED': 24, 'READY': 14, 'RUNNING': 2}
{'COMPLETED': 25, 'READY': 13, 'RUNNING': 2}
{'COMPLETED': 26, 'READY': 12, 'RUNNING': 2}
{'COMPLETED': 26, 'READY': 12, 'RUNNING': 2}
{'COMPLETED': 27, 'READY': 11, 'RUNNING': 2}
{'COMPLETED': 28, 'READY': 10, 'RUNNING': 2}
{'COMPLETED': 29, 'READY': 9, 'RUNNING': 2}
{'COMPLETED': 29, 'READY': 9, 'RUNNING': 2}
{'COMPLETED': 31, 'READY': 7, 'RUNNING': 2}
{'COMPLETED': 31, 'READY': 7, 'RUNNING': 2}
{'COMPLETED': 31, 'READY': 7, 'RUNNING': 2}
{'COMPLETED': 32, 'READY': 6, 'RUNNING': 2}
{'COMPLETED': 32, 'READY': 6, 'RUNNING': 2}
{'COMPLETED': 32, 'READY': 6, 'RUNNING': 2}
{'COMPLETED': 33, 'READY': 5, 'RUNNING': 2}
{'COMPLETED': 33, 'READY': 5, 'RUNNING': 2}
{'COMPLETED': 33, 'READY': 5, 'RUNNING': 2}
{'COMPLETED': 35, 'READY': 3, 'RUNNING': 2}
{'COMPLETED': 35, 'READY': 3, 'RUNNING': 2}
{'COMPLETED': 35, 'READY': 3, 'RUNNING': 2}
{'COMPLETED': 35, 'READY